# Stage 5 Non-VLM Baseline Sweep

Запуск no-VLM baseline: DINOv2/CLIP/SigLIP/timm features + classical classifiers. Скопируйте `drop_in/scripts/run_non_vlm_baselines.py` в `scripts/` репозитория.

In [ ]:

from pathlib import Path
import subprocess, json, os, shutil, csv
import pandas as pd
from IPython.display import display, Markdown

REPO_URL = os.environ.get("REPO_URL", "https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git")
REPO_DIR = Path(os.environ.get("REPO_DIR", "/kaggle/working/vlm-for-insulator-defect-detection"))
DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"),
    Path("/kaggle/input/idid-coco-v3"),
    Path("/kaggle/input"),
]
TRAIN_REL = Path("stage3_regrouped_v2/train_balanced/vlm_labels_v1_train_balanced_v2.annotated.jsonl")
VAL_REL = Path("stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl")

def sh(args, cwd=None, check=True):
    print("$", " ".join(str(a) for a in args))
    p = subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed {p.returncode}: {' '.join(str(a) for a in args)}")
    return p

def find_data_root():
    for root in DATASET_ROOT_CANDIDATES:
        if (root/TRAIN_REL).exists() and (root/VAL_REL).exists():
            return root
    for root in Path('/kaggle/input').glob('**') if Path('/kaggle/input').exists() else []:
        if (root/TRAIN_REL).exists() and (root/VAL_REL).exists():
            return root
    raise FileNotFoundError('Could not find stage3_regrouped_v2 train/val JSONL')


In [ ]:
gpu = sh(['nvidia-smi'], check=False)
if gpu.returncode != 0:
    print('WARNING: GPU not detected. HF/timm feature extraction may be slow.')
DATA_ROOT = find_data_root()
TRAIN_JSONL = DATA_ROOT / TRAIN_REL
VAL_JSONL = DATA_ROOT / VAL_REL
print('DATA_ROOT:', DATA_ROOT)


In [ ]:
if not REPO_DIR.exists():
    sh(['git','clone',REPO_URL,str(REPO_DIR)])
sh(['python','-m','pip','install','-q','transformers==4.57.1','accelerate','scikit-learn','pandas','numpy','pillow','timm'], cwd=REPO_DIR)


## Минимальный обязательный запуск

In [ ]:
sh([
    'python','scripts/run_non_vlm_baselines.py',
    '--train-jsonl', TRAIN_JSONL,
    '--val-jsonl', VAL_JSONL,
    '--dataset-root', DATA_ROOT,
    '--out-dir', 'reports/next_research/non_vlm_baselines',
    '--cache-dir', 'outputs/next_research/feature_cache',
    '--classifiers', 'logreg,svm',
    '--continue-on-error',
    '--skip-timm-on-import-error',
], cwd=REPO_DIR, check=True)


In [ ]:
lb_path = REPO_DIR/'reports/next_research/non_vlm_baselines/leaderboard_non_vlm.csv'
leaderboard = pd.read_csv(lb_path)
display(leaderboard)
print((REPO_DIR/'reports/next_research/non_vlm_baselines/summary.md').read_text(encoding='utf-8')[:4000])


## Архивация артефактов

In [ ]:
archive = Path('/kaggle/working/stage5_non_vlm_baselines.tar.gz')
if archive.exists(): archive.unlink()
sh(['tar','-czf',archive,'-C',REPO_DIR,'reports/next_research/non_vlm_baselines'], check=True)
print('Archive:', archive)
